# Downloading JRC GSW Yearly History — all years 2000–2022 (seasonal only)

Source: [JRC/GSW1_4/YearlyHistory](https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_YearlyHistory).

The `waterClass` band has values: 0 = no data, 1 = not water, 2 = seasonal water, 3 = permanent water.
This notebook loops over years 2000–2022 and exports a binary seasonal-water mask (`waterClass == 2`) per year.

> Note: GSW v1.4 may not cover all years up to 2022 — tasks for missing years will fail at the GEE side. Adjust `YEAR_END` if needed.

Exports go to Google Drive via Earth Engine. Monitor progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import os
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize()

In [3]:
# Define export region (global)
world_bbox = ee.Geometry.BBox(-180, -85, 180, 85)

# Year range
YEAR_START = 2000
YEAR_END   = 2022  # inclusive

In [4]:
# Loop over years and submit one seasonal-water export per year
task_log = []

for YEAR in range(YEAR_START, YEAR_END + 1):
    try:
        yearly = ee.Image(f"JRC/GSW1_4/YearlyHistory/{YEAR}").select("waterClass")

        seasonal_mask = yearly.eq(2)

        # Seasonal water export
        task_seas = ee.batch.Export.image.toDrive(
            image=seasonal_mask,
            description=f'seasonalwater_yh{YEAR}_100m_30m',
            folder='GEE_exports',
            fileNamePrefix=f'seasonalwater_yh{YEAR}_100m_30m',
            region=world_bbox,
            scale=100,
            maxPixels=1e13,
        )
        task_seas.start()
        task_log.append((YEAR, 'seasonal', task_seas.id))

        print(f"{YEAR}: submitted seasonal ({task_seas.id})")
    except Exception as e:
        print(f"{YEAR}: FAILED — {e}")

print(f"\nTotal tasks submitted: {len(task_log)}")

2000: submitted seasonal (5CUF4OZKUTYJ5NNXG4CIDDOJ)
2001: submitted seasonal (TYDUB5242AQF7UZ4VGTUJ625)
2002: submitted seasonal (J5454UA3K6SUEV7JIVBURJUJ)
2003: submitted seasonal (OAYWHOJWEHMHW4LO7VEYRGGP)
2004: submitted seasonal (CP3CVZQAU6WO2WQLV4ZUVKPI)
2005: submitted seasonal (TLE5LQEQFKG2Y2BTMYDXXXVH)
2006: submitted seasonal (WT33BA57ULOSNVBT62QTNZYW)
2007: submitted seasonal (WEYOFUGVHB37KHDBBNMZGEWX)
2008: submitted seasonal (MKSVRBD4WX4QTT3TJ53UJFQA)
2009: submitted seasonal (OERN5ZPEYHD3RMZH5FCOGJ5A)
2010: submitted seasonal (XW6I7Q6BEQON6J3UHAA7E474)
2011: submitted seasonal (H2DI6IKAGX6WZVDBKXPG4GJH)
2012: submitted seasonal (7QG432ZB7OFGPM4JKEHGPNHY)
2013: submitted seasonal (KIIUTR5TUTKE6FZDTMYLGZ4O)
2014: submitted seasonal (YKWBNIWHWQDBHY3CMEOSCGHY)
2015: submitted seasonal (5VBFAHTTZV34TUU2POMAR6LC)
2016: submitted seasonal (SDEP7T3ZRHME5BJWR464C7M4)
2017: submitted seasonal (EGCO4LGRTFJKOORVB2AQ3RAA)
2018: submitted seasonal (NVBWEBVXVOWYOO5XCZQV3S5X)
2019: submit

In [5]:
# Optional: save the task log for tracking
df_tasks = pd.DataFrame(task_log, columns=["year", "type", "task_id"])
df_tasks

,year,type,task_id
0,2000,seasonal,5CUF4OZKUTYJ5NNXG4CIDDOJ
1,2001,seasonal,TYDUB5242AQF7UZ4VGTUJ625
2,2002,seasonal,J5454UA3K6SUEV7JIVBURJUJ
3,2003,seasonal,OAYWHOJWEHMHW4LO7VEYRGGP
4,2004,seasonal,CP3CVZQAU6WO2WQLV4ZUVKPI
5,2005,seasonal,TLE5LQEQFKG2Y2BTMYDXXXVH
6,2006,seasonal,WT33BA57ULOSNVBT62QTNZYW
7,2007,seasonal,WEYOFUGVHB37KHDBBNMZGEWX
8,2008,seasonal,MKSVRBD4WX4QTT3TJ53UJFQA
9,2009,seasonal,OERN5ZPEYHD3RMZH5FCOGJ5A


### NOTE
- Track export progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks).
- After downloads finish, place the tiles under `Measures_work/maps/raw/Water_surface/seasonalwater_yh{YEAR}/` so a downstream `*_ethnologue.ipynb` can glob them by folder.
- 1 task/year × 23 years = ~23 export tasks. GEE typically allows ~3000 concurrent tasks, so this fits comfortably.